# frametime-lite (Colab notebook)

Analyse a frame-time capture (CSV) from a game or 3D demo: average FPS, percentiles, 1% / 0.1% lows, stutter detection, charts, and a before/after comparison.

**What you need:** a CSV file with a header row and one frame time (in milliseconds) per row, e.g. exported by a tool like PresentMon or NVIDIA FrameView. No capture yet? Run cells 1 and 4 to try the demo.

**How to use it**
1. Run **cell 1** (defines the functions; it prints nothing, that is normal).
2. Run **cell 2** to analyse one capture, and/or **cell 3** to compare two captures (before / after).
3. Or run **cell 4** for a demo that needs no file.

**Good to know**
- Everything stays in memory: nothing is written to your Google Drive. Uploaded files disappear when the session ends.
- The demo (cell 4) uses **synthetic data** (random numbers), not real measurements.
- Learning project built with the help of an AI assistant (Claude), then reviewed and tested. It is a simple statistics tool, not a profiler: it tells you *that* frame pacing got worse, not *why*.

## 1. The tool (run once)

In [ ]:
import csv, io
import numpy as np
import matplotlib.pyplot as plt

# Column names tried in order when COLUMN is not set (check them against your own files!).
COMMON_NAMES = ["MsBetweenPresents", "FrameTime", "frametime"]


def load_frame_times(text, column=None):
    """Parse CSV text and return the frame times (in ms) as a numpy array.

    Rows that are not numeric or not > 0 are skipped.
    """
    reader = csv.DictReader(io.StringIO(text.lstrip("\ufeff")))
    if not reader.fieldnames:
        raise ValueError("The file is empty")
    if column is None:
        column = next((n for n in COMMON_NAMES if n in reader.fieldnames), None)
        if column is None:
            raise ValueError(f"No frame-time column found. Columns in the file: {reader.fieldnames}. "
                             "Set COLUMN to the right name.")
    elif column not in reader.fieldnames:
        raise ValueError(f"Column {column!r} not found. Columns in the file: {reader.fieldnames}")
    values = []
    for row in reader:
        try:
            v = float(row[column])
        except (TypeError, ValueError):
            continue  # skip rows that are not numbers
        if v > 0:
            values.append(v)
    if not values:
        raise ValueError("No valid frame times found in that column")
    return np.array(values)


def low_fps(ft, fraction):
    """FPS equivalent of the mean of the slowest `fraction` of frames (0.01 = 1% low)."""
    k = max(1, int(np.ceil(len(ft) * fraction - 1e-9)))  # at least one frame
    return 1000.0 / np.sort(ft)[-k:].mean()


def compute_stats(ft):
    """Return a dict of summary metrics for an array of frame times (ms)."""
    return {
        "frames": len(ft),
        "duration_s": ft.sum() / 1000.0,
        "avg_fps": len(ft) * 1000.0 / ft.sum(),  # frames / total time
        "mean_ms": ft.mean(),
        "median_ms": np.median(ft),
        "p95_ms": np.percentile(ft, 95),
        "p99_ms": np.percentile(ft, 99),
        "p999_ms": np.percentile(ft, 99.9),
        "low1_fps": low_fps(ft, 0.01),
        "low01_fps": low_fps(ft, 0.001),
    }


def find_stutters(ft, factor=2.0, window=61, min_excess_ms=5.0):
    """Find stutters: frames slower than `factor` x the local (rolling) median
    AND at least `min_excess_ms` above it.

    Returns a list of (frame_index, start_time_s, frame_time_ms).
    """
    half = window // 2
    start_times = (np.cumsum(ft) - ft) / 1000.0
    found = []
    for i in range(len(ft)):
        local = np.median(ft[max(0, i - half): i + half + 1])
        if ft[i] > factor * local and ft[i] - local >= min_excess_ms:
            found.append((i, start_times[i], ft[i]))
    return found


def print_report(stats, stutters):
    """Print the summary metrics and the first stutters."""
    print(f"Frames: {stats['frames']}   Duration: {stats['duration_s']:.1f} s")
    print(f"Average FPS: {stats['avg_fps']:.1f}")
    print(f"Frame time (ms): mean {stats['mean_ms']:.2f}, median {stats['median_ms']:.2f}")
    print(f"Percentiles (ms): p95 {stats['p95_ms']:.2f}, p99 {stats['p99_ms']:.2f}, p99.9 {stats['p999_ms']:.2f}")
    print(f"1% low: {stats['low1_fps']:.1f} FPS   0.1% low: {stats['low01_fps']:.1f} FPS")
    print(f"Stutters: {len(stutters)}")
    for i, t, ms in stutters[:20]:
        print(f"  frame {i} at {t:.2f} s: {ms:.1f} ms")
    if len(stutters) > 20:
        print(f"  ... and {len(stutters) - 20} more")


def compare(base, cand, threshold_pct=5.0):
    """Print a baseline -> candidate table. Return True if any metric regressed.

    A metric regresses when it gets worse by more than `threshold_pct` percent
    (lower FPS, or higher frame time).
    """
    higher_is_better = {"avg_fps", "low1_fps", "low01_fps"}
    regressed = False
    for k in ["avg_fps", "median_ms", "p95_ms", "p99_ms", "p999_ms", "low1_fps", "low01_fps"]:
        change = (cand[k] - base[k]) / base[k] * 100
        worse = -change if k in higher_is_better else change
        flag = ""
        if worse > threshold_pct:
            flag = "  <-- REGRESSION"
            regressed = True
        print(f"{k:10s} {base[k]:9.2f} -> {cand[k]:9.2f}  ({change:+.1f}%){flag}")
    return regressed


def show_plots(series, stutters_by_name=None):
    """Show three charts. `series` maps a name to an array of frame times (ms)."""
    stutters_by_name = stutters_by_name or {}

    # 1) Frame time over time (stutters in red)
    plt.figure(figsize=(10, 3.5))
    for name, ft in series.items():
        t = (np.cumsum(ft) - ft) / 1000.0
        plt.plot(t, ft, linewidth=0.6, label=name)
        st = stutters_by_name.get(name)
        if st:
            plt.scatter([s[1] for s in st], [s[2] for s in st], color="red", s=12)
    plt.xlabel("Time (s)"); plt.ylabel("Frame time (ms)"); plt.legend(); plt.title("Frame times over time")
    plt.show()

    # 2) Histogram (log scale so rare slow frames stay visible)
    plt.figure(figsize=(6, 3.5))
    for name, ft in series.items():
        plt.hist(ft, bins=100, alpha=0.6, label=name)
    plt.yscale("log"); plt.xlabel("Frame time (ms)"); plt.ylabel("Frames (log scale)")
    plt.legend(); plt.title("Frame-time histogram"); plt.show()

    # 3) Percentile curve (x axis stretched so the slow tail is readable)
    plt.figure(figsize=(6, 3.5))
    for name, ft in series.items():
        s = np.sort(ft)
        p = (np.arange(len(s)) + 0.5) / len(s)
        keep = p >= 0.5
        plt.plot(-np.log10(1 - p[keep]), s[keep], label=name)
    plt.xticks([0.30103, 1, 2, 3], ["50", "90", "99", "99.9"])
    plt.xlabel("Percentile"); plt.ylabel("Frame time (ms)"); plt.legend()
    plt.title("Frame time by percentile"); plt.show()

## 2. Analyse one capture
A file picker opens: choose your CSV (frame times in **ms**).
If the column is not detected automatically, set `COLUMN` to its exact name (the error message lists the columns found in your file).

In [ ]:
from google.colab import files

COLUMN = None   # e.g. "MsBetweenPresents"; None = auto-detect

uploaded = files.upload()
name, content = next(iter(uploaded.items()))
ft = load_frame_times(content.decode("utf-8-sig"), COLUMN)
stats = compute_stats(ft)
stutters = find_stutters(ft)
print_report(stats, stutters)
show_plots({name: ft}, {name: stutters})

## 3. Compare two captures (optional)
Pick the **baseline** (before) file first, then the **candidate** (after) file.
A metric is flagged as a regression when it is worse than the baseline by more than `THRESHOLD_PCT` percent.
Keep in mind that two captures of the *same* build already differ a little (run-to-run variance), so repeat captures before trusting a small difference.

In [ ]:
print("First choose the BASELINE (before) file, then the CANDIDATE (after) file.")
up_a = files.upload()
up_b = files.upload()
(name_a, txt_a), (name_b, txt_b) = next(iter(up_a.items())), next(iter(up_b.items()))
ft_a = load_frame_times(txt_a.decode("utf-8-sig"), COLUMN)
ft_b = load_frame_times(txt_b.decode("utf-8-sig"), COLUMN)

THRESHOLD_PCT = 5.0
regressed = compare(compute_stats(ft_a), compute_stats(ft_b), THRESHOLD_PCT)
print("\n=> Regression detected" if regressed else "\n=> No regression above the threshold")
show_plots({"baseline": ft_a, "candidate": ft_b},
           {"baseline": find_stutters(ft_a), "candidate": find_stutters(ft_b)})

## 4. Demo on synthetic data
Random numbers generated here, **not** from a real game. Series A runs at about 16.7 ms per frame (~60 FPS) with one frame deliberately set to 90 ms (a stutter). Series B is steadily slower (~19 ms) with no stutter.

In [ ]:
# SYNTHETIC DATA (random numbers, not from a real game)
rng = np.random.default_rng(0)
demo_a = rng.normal(16.7, 0.5, 3000); demo_a[1000] = 90   # one injected 90 ms stutter
demo_b = rng.normal(19.0, 0.5, 3000)                      # slower, but steady
print_report(compute_stats(demo_a), find_stutters(demo_a))
compare(compute_stats(demo_a), compute_stats(demo_b))
show_plots({"synthetic A": demo_a, "synthetic B": demo_b})